Gian Tituaña, 325991.


## Instalación de jar de postgres

In [2]:
# Instalación de dependencias y descarga de JARs para PostgreSQL

# Crear directorio para JARs si no existe
import os
jars_dir = './work/jars'
os.makedirs(jars_dir, exist_ok=True)

# Descargar JARs necesarios para Spark 3.x con Scala 2.12
postgresql_jdbc_url = "https://repo1.maven.org/maven2/org/postgresql/postgresql/42.7.3/postgresql-42.7.3.jar"

# Descargar JARs
!wget -O {jars_dir}/postgresql-42.7.3.jar {postgresql_jdbc_url}

print("JARs descargados exitosamente")

JARs descargados exitosamente


--2025-11-08 11:14:36--  https://repo1.maven.org/maven2/org/postgresql/postgresql/42.7.3/postgresql-42.7.3.jar
Resolving repo1.maven.org (repo1.maven.org)... 2606:4700::6812:130c, 2606:4700::6812:120c, 104.18.18.12, ...
Connecting to repo1.maven.org (repo1.maven.org)|2606:4700::6812:130c|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1089312 (1,0M) [application/java-archive]
Saving to: './work/jars/postgresql-42.7.3.jar'

     0K .......... .......... .......... .......... ..........  4% 11,9M 0s
    50K .......... .......... .......... .......... ..........  9% 5,46M 0s
   100K .......... .......... .......... .......... .......... 14% 6,22M 0s
   150K .......... .......... .......... .......... .......... 18% 40,4M 0s
   200K .......... .......... .......... .......... .......... 23% 6,20M 0s
   250K .......... .......... .......... .......... .......... 28% 12,7M 0s
   300K .......... .......... .......... .......... .......... 32% 81,1M 0s
   350K .......

## Creacion de la sesion


In [1]:
# INICIALIZAR SPARK SESSION - VERSIÓN ROBUSTA
print("=== INICIALIZANDO SPARK CON POSTGRESQL ===")

import sys
import os
os.environ["PYSPARK_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T

# Configuración de JARs para PostgreSQL
jars_dir = './work/jars'
jar_path = f"{jars_dir}/postgresql-42.7.3.jar"

# Verificar que el JAR existe
if not os.path.isfile(jar_path):
    raise FileNotFoundError(f"JAR no encontrado: {jar_path}")

print(f"Usando JAR: {jar_path}")

# DETENER sesión existente si existe
try:
    spark.stop()
    print("Sesión anterior detenida")
except:
    pass

# Crear NUEVA sesión Spark con configuración robusta
spark = SparkSession.builder \
    .appName("NYC_Taxi_Ingesta_PostgreSQL") \
    .config("spark.jars", jar_path) \
    .config("spark.driver.extraClassPath", jar_path) \
    .config("spark.executor.extraClassPath", jar_path) \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .config("spark.sql.execution.pyspark.udf.faulthandler.enabled", "true") \
    .config("spark.python.worker.faulthandler.enabled", "true") \
    .getOrCreate()

# Verificar que funciona
test_count = spark.range(5).count()
print(f"✓ Spark funcionando correctamente - Test: {test_count} registros")
print(f"✓ Versión Spark: {spark.version}")
print(f"✓ PostgreSQL JAR configurado")

# Verificar que el driver está disponible
try:
    # Intento de carga del driver
    spark._jvm.Class.forName("org.postgresql.Driver")
    print("✓ Driver PostgreSQL cargado exitosamente")
except Exception as e:
    print(f"✗ Error cargando driver: {e}")

=== INICIALIZANDO SPARK CON POSTGRESQL ===
Usando JAR: ./work/jars/postgresql-42.7.3.jar
✓ Spark funcionando correctamente - Test: 5 registros
✓ Versión Spark: 4.0.1
✓ PostgreSQL JAR configurado
✓ Driver PostgreSQL cargado exitosamente


In [ ]:
# Descarga archivos Parquet y CSV de taxi NYC
import os
from pathlib import Path

start_year = 2015
end_year = 2025
months = range(1, 13)
base_url = 'https://d37ci6vzurychx.cloudfront.net/trip-data'
data_dir = './work/data'  # Carpeta local para guardar los archivos
os.makedirs(data_dir, exist_ok=True)
missing_files = []

for year in range(start_year, end_year + 1):
    for color in ['yellow','green']:
        for m in months:
            fname = f'{color}_tripdata_{year}-{m:02d}.parquet'
            url = f'{base_url}/{fname}'
            dest = f'{data_dir}/{fname}'
            print(f'Descargando {fname} ...')
            exit_code = os.system(f'wget -O "{dest}" "{url}"')
            if exit_code != 0 or not Path(dest).is_file():
                print(f'ERROR: No se pudo descargar {fname}')
                missing_files.append(fname)

# Resumen
if missing_files:
    print('Faltan los siguientes archivos:')
    for f in missing_files:
        print('-', f)
else:
    print('Todos los archivos descargados correctamente.')

Descargando yellow_tripdata_2015-01.parquet ...


## Ingesta taxi zones

In [ ]:
# INGESTA TAXI_ZONE_LOOKUP A POSTGRESQL
import os
import time
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql import types as T
import psycopg2

# ============================================================================
# CONFIGURACIÓN POSTGRESQL DESDE VARIABLES DE ENTORNO
# ============================================================================
PG_HOST = os.getenv('PG_HOST', 'localhost')
PG_PORT = int(os.getenv('PG_PORT', 5432))
PG_DB = os.getenv('PG_DB', 'postgres')
PG_USER = os.getenv('PG_USER', 'postgres')
PG_PASSWORD = os.getenv('PG_PASSWORD', 'root')
PG_SCHEMA_RAW = os.getenv('PG_SCHEMA_RAW', 'raw')

print(f" Conectando a PostgreSQL:")
print(f"   Host: {PG_HOST}:{PG_PORT}")
print(f"   Database: {PG_DB}")
print(f"   User: {PG_USER}")
print(f"   Schema: {PG_SCHEMA_RAW}")

# Configuración PostgreSQL para Spark JDBC
pgOptions = {
    "url": f"jdbc:postgresql://{PG_HOST}:{PG_PORT}/{PG_DB}",
    "driver": "org.postgresql.Driver",
    "user": PG_USER,
    "password": PG_PASSWORD,
    "dbtable": f"{PG_SCHEMA_RAW}.taxi_zone_lookup"
}

# Crear esquema/tabla si no existe (seguro y idempotente)
postgres_schema_zones = f'''
CREATE SCHEMA IF NOT EXISTS {PG_SCHEMA_RAW};
CREATE TABLE IF NOT EXISTS {PG_SCHEMA_RAW}.taxi_zone_lookup (
    locationid INT,
    borough VARCHAR,
    zone VARCHAR,
    service_zone VARCHAR
);
'''

try:
    conn = psycopg2.connect(
        dbname=PG_DB,
        user=PG_USER,
        password=PG_PASSWORD,
        host=PG_HOST,
        port=PG_PORT
    )
    conn.autocommit = True
    cur = conn.cursor()
    cur.execute(postgres_schema_zones)
    cur.close()
    conn.close()
    print(f"✓ Schema/table {PG_SCHEMA_RAW}.taxi_zone_lookup creada o ya existe.")
except Exception as e:
    print(f"ERROR creando schema/table {PG_SCHEMA_RAW}.taxi_zone_lookup: {e}")

# Configuración para taxi zones
data_dir = './work/data'
zone_file = 'taxi_zone_lookup.csv'
zone_url = 'https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv'
zone_path = os.path.join(data_dir, zone_file)
run_id = f"taxi_zones_pg_{datetime.utcnow().strftime('%Y%m%d_%H%M%S')}"
ingested_at_utc = datetime.utcnow().isoformat()

print(f"INICIANDO INGESTA TAXI ZONES a PostgreSQL")
print(f"Run ID: {run_id}")
print(f"Archivo: {zone_file}")

# Verificar si el directorio de datos existe, si no, crearlo
os.makedirs(data_dir, exist_ok=True)

# Descargar el archivo si no existe
if not os.path.isfile(zone_path):
    print(f"Descargando archivo {zone_file}...")
    exit_code = os.system(f'wget -O {zone_path} {zone_url}')
    if exit_code != 0:
        print(f"ERROR: Falló la descarga del archivo {zone_file}")
        # Puedes agregar un raise o sys.exit() aquí si la descarga es crítica
else:
    print(f"Archivo {zone_file} ya existe en {data_dir}")

# Verificar que el archivo existe después de intentar descargar
if not os.path.isfile(zone_path):
    print(f"ERROR: Archivo {zone_file} no encontrado en {data_dir} después de descarga")
else:
    try:
        start_time = time.time()

        # PASO 1: Leer archivo CSV
        print("Leyendo archivo taxi_zone_lookup.csv...")
        df_zones = spark.read.option("header", "true").option("inferSchema", "true").csv(zone_path)

        # Mostrar esquema y sample
        print("Esquema detectado:")
        df_zones.printSchema()

        # Cache para evitar re-lecturas
        df_zones.cache()
        total_zones = df_zones.count()
        print(f"Total zonas encontradas: {total_zones:,}")

        if total_zones == 0:
            print("ADVERTENCIA: Archivo vacío")
        else:
            # Mostrar muestra de datos
            print("\nMuestra de datos:")
            df_zones.show(5, truncate=False)

            # PASO 2: Transformar datos con casting explícito
            print("Aplicando transformaciones de tipos...")

            # Solo datos originales con casting para consistencia
            df_zones_final = df_zones.select(
                F.col('LocationID').cast(T.IntegerType()).alias('locationid'), # Nombres en minúscula para PostgreSQL
                F.col('Borough').cast(T.StringType()).alias('borough'),
                F.col('Zone').cast(T.StringType()).alias('zone'),
                F.col('service_zone').cast(T.StringType()).alias('service_zone')
            )

            # Cache del DataFrame final
            df_zones_final.cache()
            df_zones.unpersist()  # Liberar original

            # Verificar datos finales
            final_count = df_zones_final.count()
            print(f"Registros después de transformación: {final_count:,}")

            # PASO 3: Cargar a PostgreSQL
            table_name = pgOptions["dbtable"]
            print(f"Cargando {final_count:,} registros a tabla {table_name} en PostgreSQL...")

            # Escribir a PostgreSQL
            (
                df_zones_final.write
                .format("jdbc")
                .options(**pgOptions)
                .mode("overwrite")
                .save()
            )

            # Limpieza de memoria
            df_zones_final.unpersist()

            # Métricas finales
            processing_time = time.time() - start_time

            print(f"\n{'='*50}")
            print(f"INGESTA TAXI ZONES a PostgreSQL COMPLETADA")
            print(f"{'='*50}")
            print(f"Registros procesados: {final_count:,}")
            print(f"Tiempo total: {processing_time:.2f} segundos")
            print(f"Velocidad: {final_count/processing_time:,.0f} registros/segundo")
            print(f"Tabla: {table_name} (OVERWRITE)")
            print(f"Estado: ÉXITO")

    except Exception as e:
        print(f"ERROR en ingesta de taxi zones a PostgreSQL: {e}")
        # Limpieza en caso de error
        for var_name in ['df_zones', 'df_zones_final']:
            try:
                if var_name in locals():
                    locals()[var_name].unpersist()
            except:
                pass


✓ Schema/table raw.taxi_zone_lookup creada o ya existe.
INICIANDO INGESTA TAXI ZONES a PostgreSQL
Run ID: taxi_zones_pg_20251107_210914
Archivo: taxi_zone_lookup.csv
Descargando archivo taxi_zone_lookup.csv...


C:\Users\marti\AppData\Local\Temp\ipykernel_21912\3392782621.py:45: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  run_id = f"taxi_zones_pg_{datetime.utcnow().strftime('%Y%m%d_%H%M%S')}"
C:\Users\marti\AppData\Local\Temp\ipykernel_21912\3392782621.py:46: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ingested_at_utc = datetime.utcnow().isoformat()


Leyendo archivo taxi_zone_lookup.csv...
Esquema detectado:
root
 |-- LocationID: integer (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)

Esquema detectado:
root
 |-- LocationID: integer (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)

Total zonas encontradas: 265

Muestra de datos:
Total zonas encontradas: 265

Muestra de datos:
+----------+-------------+-----------------------+------------+
|LocationID|Borough      |Zone                   |service_zone|
+----------+-------------+-----------------------+------------+
|1         |EWR          |Newark Airport         |EWR         |
|2         |Queens       |Jamaica Bay            |Boro Zone   |
|3         |Bronx        |Allerton/Pelham Gardens|Boro Zone   |
|4         |Manhattan    |Alphabet City          |Yellow Zone |
|5         |Staten Island|Arden Heights   

##  Ingesta Green


In [ ]:
# INGESTA GREEN TAXI A POSTGRESQL - Con particionamiento automático
import os
import time
from datetime import datetime, timezone
from pyspark.sql import functions as F
from pyspark.sql import types as T
import psycopg2

# ============================================================================
# CONFIGURACIÓN POSTGRESQL DESDE VARIABLES DE ENTORNO
# ============================================================================
PG_HOST = os.getenv('PG_HOST', 'localhost')
PG_PORT = int(os.getenv('PG_PORT', 5432))
PG_DB = os.getenv('PG_DB', 'postgres')
PG_USER = os.getenv('PG_USER', 'postgres')
PG_PASSWORD = os.getenv('PG_PASSWORD', 'root')
PG_SCHEMA_RAW = os.getenv('PG_SCHEMA_RAW', 'raw')

print(f"📊 Conectando a PostgreSQL:")
print(f"   Host: {PG_HOST}:{PG_PORT}")
print(f"   Database: {PG_DB}")
print(f"   User: {PG_USER}")
print(f"   Schema: {PG_SCHEMA_RAW}")

# Configuración
service_types = ['green']
start_year = 2023
end_year = 2025
months = [1,2,3,4,5,6,7,8,9,10,11,12]
data_dir = './work/data'

run_id = f"raw_green_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}"
ingested_at_utc = datetime.now(timezone.utc).isoformat()

# Configuración PostgreSQL JDBC
pgOptions = {
    "url": f"jdbc:postgresql://{PG_HOST}:{PG_PORT}/{PG_DB}",
    "driver": "org.postgresql.Driver",
    "user": PG_USER,
    "password": PG_PASSWORD,
    "dbtable": f"{PG_SCHEMA_RAW}.green_taxi_trip"
}

# Función para crear partición si no existe
def create_partition_if_not_exists(year, month):
    """Crea la partición para año-mes si no existe"""
    partition_name = f"green_taxi_trip_y{year}m{month:02d}"
    
    # Calcular límites de la partición
    next_month = month + 1
    next_year = year
    if next_month > 12:
        next_month = 1
        next_year += 1
    
    create_partition_sql = f"""
    CREATE TABLE IF NOT EXISTS {PG_SCHEMA_RAW}.{partition_name} 
    PARTITION OF {PG_SCHEMA_RAW}.green_taxi_trip
    FOR VALUES FROM ({year}, {month}) TO ({next_year}, {next_month});
    """
    
    try:
        conn = psycopg2.connect(
            dbname=PG_DB,
            user=PG_USER,
            password=PG_PASSWORD,
            host=PG_HOST,
            port=PG_PORT
        )
        conn.autocommit = True
        cur = conn.cursor()
        cur.execute(create_partition_sql)
        cur.close()
        conn.close()
        print(f"   ✓ Partición {partition_name} creada o ya existe")
        return True
    except Exception as e:
        print(f"   ERROR creando partición {partition_name}: {e}")
        return False

# Esquema de la tabla principal PARTICIONADA
postgres_schema = f'''
CREATE SCHEMA IF NOT EXISTS {PG_SCHEMA_RAW};

-- Eliminar tabla existente si no está particionada (para migración limpia)
DO $$ 
BEGIN
    IF EXISTS (
        SELECT 1 FROM pg_tables 
        WHERE schemaname = '{PG_SCHEMA_RAW}' AND tablename = 'green_taxi_trip'
    ) AND NOT EXISTS (
        SELECT 1 FROM pg_partitioned_table pt
        JOIN pg_class c ON pt.partrelid = c.oid
        WHERE c.relname = 'green_taxi_trip'
    ) THEN
        DROP TABLE {PG_SCHEMA_RAW}.green_taxi_trip CASCADE;
        RAISE NOTICE 'Tabla antigua no particionada eliminada';
    END IF;
END $$;

CREATE TABLE IF NOT EXISTS {PG_SCHEMA_RAW}.green_taxi_trip (
    run_id VARCHAR,
    service_type VARCHAR,
    source_year INT NOT NULL,
    source_month INT NOT NULL,
    ingested_at_utc VARCHAR,
    source_path VARCHAR,
    VendorID INT,
    lpep_pickup_datetime TIMESTAMP,
    lpep_dropoff_datetime TIMESTAMP,
    store_and_fwd_flag VARCHAR,
    RatecodeID INT,
    PULocationID INT,
    DOLocationID INT,
    passenger_count INT,
    trip_distance FLOAT,
    fare_amount FLOAT,
    extra FLOAT,
    mta_tax FLOAT,
    tip_amount FLOAT,
    tolls_amount FLOAT,
    ehail_fee INT,
    improvement_surcharge FLOAT,
    total_amount FLOAT,
    payment_type INT,
    trip_type FLOAT,
    congestion_surcharge FLOAT,
    cbd_congestion_fee FLOAT
) PARTITION BY RANGE (source_year, source_month);

-- Crear índice en la tabla particionada (se propagará a las particiones)
CREATE INDEX IF NOT EXISTS idx_green_taxi_year_month 
ON {PG_SCHEMA_RAW}.green_taxi_trip (source_year, source_month);
'''

# Crear la tabla particionada
try:
    conn = psycopg2.connect(
        dbname=PG_DB,
        user=PG_USER,
        password=PG_PASSWORD,
        host=PG_HOST,
        port=PG_PORT
    )
    conn.autocommit = True
    cur = conn.cursor()
    cur.execute(postgres_schema)
    cur.close()
    conn.close()
    print(f"✓ Tabla particionada {PG_SCHEMA_RAW}.green_taxi_trip creada o ya existe")
except Exception as e:
    print(f"ERROR creando tabla: {e}")
    exit(1)

# Crear todas las particiones necesarias de antemano
print("\nCreando particiones necesarias...")
for year in range(start_year, end_year + 1):
    for month in months:
        create_partition_if_not_exists(year, month)

# Métricas globales
total_files_processed = 0
total_files_skipped = 0
total_records_read = 0
total_records_normalized = 0
total_records_transformed = 0
total_records_written = 0
total_records_deleted = 0
total_processing_time = 0

for service_type in service_types:
    print(f"\n{'='*80}")
    print(f"PROCESANDO {service_type.upper()} TAXI")
    print(f"{'='*80}")
    
    for year in range(start_year, end_year + 1):
        for month in months:
            fname = f'{service_type}_tripdata_{year}-{month:02d}.parquet'
            fpath = os.path.join(data_dir, fname)
            print(f"\nArchivo: {fname}")
            
            if not os.path.isfile(fpath):
                print(f"   Archivo no encontrado: {fname}")
                total_files_skipped += 1
                continue
            
            file_start_time = time.time()
            
            try:
                # PASO 1: Eliminar registros existentes del año/mes (idempotencia)
                print(f"   Verificando registros existentes para {year}-{month:02d}...")
                try:
                    conn = psycopg2.connect(
                        dbname=PG_DB,
                        user=PG_USER,
                        password=PG_PASSWORD,
                        host=PG_HOST,
                        port=PG_PORT
                    )
                    conn.autocommit = True
                    cur = conn.cursor()

                    # Contar registros existentes
                    cur.execute(
                        f"SELECT COUNT(*) FROM {PG_SCHEMA_RAW}.green_taxi_trip WHERE source_year = %s AND source_month = %s", 
                        (year, month)
                    )
                    existing_count = cur.fetchone()[0]
                    
                    if existing_count > 0:
                        print(f"   ⚠ Encontrados {existing_count:,} registros existentes - eliminando...")
                        cur.execute(
                            f"DELETE FROM {PG_SCHEMA_RAW}.green_taxi_trip WHERE source_year = %s AND source_month = %s", 
                            (year, month)
                        )
                        print(f"   ✓ {existing_count:,} registros eliminados")
                        total_records_deleted += existing_count
                    else:
                        print(f"   ✓ No hay registros previos para este periodo")
                    
                    cur.close()
                    conn.close()
                except Exception as e:
                    print(f"   ERROR verificando/eliminando registros: {e}")
                
                # PASO 2: Lectura del archivo
                df = spark.read.option("mergeSchema", "false").parquet(fpath)
                
                records_read = df.count()
                print(f"   Registros leídos: {records_read:,}")
                
                if records_read == 0:
                    print("   Archivo vacío, saltando...")
                    total_files_skipped += 1
                    continue
                
                records_normalized = records_read
                pickup_col = 'lpep_pickup_datetime'
                dropoff_col = 'lpep_dropoff_datetime'
                has_cbd_fee = 'cbd_congestion_fee' in df.columns
                
                # PASO 3: Transformaciones
                df_transformed = df.select(
                    F.lit(run_id).alias('run_id'),
                    F.lit(service_type).alias('service_type'),
                    F.lit(year).alias('source_year'),
                    F.lit(month).alias('source_month'),
                    F.lit(ingested_at_utc).alias('ingested_at_utc'),
                    F.lit(fpath).alias('source_path'),
                    F.col('VendorID').cast(T.IntegerType()).alias('VendorID'),
                    F.col(pickup_col).cast(T.TimestampType()).alias('lpep_pickup_datetime'),
                    F.col(dropoff_col).cast(T.TimestampType()).alias('lpep_dropoff_datetime'),
                    F.col('store_and_fwd_flag').cast(T.StringType()).alias('store_and_fwd_flag'),
                    F.col('RatecodeID').cast(T.IntegerType()).alias('RatecodeID'),
                    F.col('PULocationID').cast(T.IntegerType()).alias('PULocationID'),
                    F.col('DOLocationID').cast(T.IntegerType()).alias('DOLocationID'),
                    F.col('passenger_count').cast(T.IntegerType()).alias('passenger_count'),
                    F.col('trip_distance').cast(T.FloatType()).alias('trip_distance'),
                    F.col('fare_amount').cast(T.FloatType()).alias('fare_amount'),
                    F.col('extra').cast(T.FloatType()).alias('extra'),
                    F.col('mta_tax').cast(T.FloatType()).alias('mta_tax'),
                    F.col('tip_amount').cast(T.FloatType()).alias('tip_amount'),
                    F.col('tolls_amount').cast(T.FloatType()).alias('tolls_amount'),
                    F.col('ehail_fee').cast(T.IntegerType()).alias('ehail_fee'),
                    F.col('improvement_surcharge').cast(T.FloatType()).alias('improvement_surcharge'),
                    F.col('total_amount').cast(T.FloatType()).alias('total_amount'),
                    F.col('payment_type').cast(T.IntegerType()).alias('payment_type'),
                    F.col('trip_type').cast(T.FloatType()).alias('trip_type'),
                    F.col('congestion_surcharge').cast(T.FloatType()).alias('congestion_surcharge'),
                    (F.col('cbd_congestion_fee').cast(T.FloatType()) if has_cbd_fee 
                     else F.lit(None).cast(T.FloatType())).alias('cbd_congestion_fee')
                )
                
                records_transformed = df_transformed.count()
                print(f"   Registros transformados: {records_transformed:,}")
                
                # PASO 4: Escritura JDBC a la partición correspondiente
                print(f"   Escribiendo a partición y{year}m{month:02d}...")
                (
                    df_transformed.write
                    .format("jdbc")
                    .options(**pgOptions)
                    .option("batchsize", 10000)
                    .option("numPartitions", 4)
                    .option("isolationLevel", "READ_UNCOMMITTED")
                    .mode("append")
                    .save()
                )
                
                df_transformed.unpersist()
                df.unpersist()
                
                records_written = records_transformed
                file_processing_time = time.time() - file_start_time
                
                # Actualizar métricas
                total_files_processed += 1
                total_records_read += records_read
                total_records_normalized += records_normalized
                total_records_transformed += records_transformed
                total_records_written += records_written
                total_processing_time += file_processing_time
                
                print(f"   ✓ Completado en {file_processing_time:.2f}s")
                print(f"   Velocidad: {records_written/file_processing_time:,.0f} reg/s")
                
                if records_read != records_written:
                    lost_records = records_read - records_written
                    loss_pct = (lost_records / records_read) * 100
                    print(f"   ⚠ Registros perdidos: {lost_records:,} ({loss_pct:.2f}%)")
                
                # Limpieza preventiva de caché
                if total_files_processed % 3 == 0:
                    spark.catalog.clearCache()
                    print("   Cache limpiado")
                    
            except Exception as e:
                print(f"   ERROR procesando {fname}: {e}")
                import traceback
                traceback.print_exc()
                total_files_skipped += 1
                continue

# RESUMEN FINAL
print(f"\n{'='*80}")
print(f"RESUMEN FINAL - GREEN TAXI PARTICIONADO")
print(f"{'='*80}")
print(f"Run ID: {run_id}")
print(f"Timestamp UTC: {ingested_at_utc}")
print(f"\nCONFIGURACIÓN:")
print(f"   Método: PostgreSQL JDBC con particionamiento RANGE(year, month)")
print(f"   Particiones: {(end_year - start_year + 1) * len(months)} creadas")
print(f"   Paralelismo: 4 particiones Spark")
print(f"   Batch size: 10,000 registros")
print(f"\nARCHIVOS:")
print(f"   Procesados: {total_files_processed}")
print(f"   Omitidos: {total_files_skipped}")
print(f"   Total: {total_files_processed + total_files_skipped}")
print(f"\nREGISTROS:")
print(f"   Eliminados (previos): {total_records_deleted:,}")
print(f"   Leídos: {total_records_read:,}")
print(f"   Transformados: {total_records_transformed:,}")
print(f"   Escritos: {total_records_written:,}")
print(f"\nRENDIMIENTO:")
if total_processing_time > 0:
    print(f"   Tiempo total: {total_processing_time:.2f}s ({total_processing_time/60:.2f} min)")
    print(f"   Velocidad: {total_records_written/total_processing_time:,.0f} reg/s")
print(f"{'='*80}")


✓ Tabla particionada raw.green_taxi_trip creada o ya existe

Creando particiones necesarias...
   ✓ Partición green_taxi_trip_y2023m01 creada o ya existe
   ✓ Partición green_taxi_trip_y2023m02 creada o ya existe
   ✓ Partición green_taxi_trip_y2023m03 creada o ya existe
   ✓ Partición green_taxi_trip_y2023m04 creada o ya existe
   ✓ Partición green_taxi_trip_y2023m05 creada o ya existe
   ✓ Partición green_taxi_trip_y2023m06 creada o ya existe
   ✓ Partición green_taxi_trip_y2023m07 creada o ya existe
   ✓ Partición green_taxi_trip_y2023m08 creada o ya existe
   ✓ Partición green_taxi_trip_y2023m09 creada o ya existe
   ✓ Partición green_taxi_trip_y2023m10 creada o ya existe
   ✓ Partición green_taxi_trip_y2023m11 creada o ya existe
   ✓ Partición green_taxi_trip_y2023m12 creada o ya existe
   ✓ Partición green_taxi_trip_y2024m01 creada o ya existe
   ✓ Partición green_taxi_trip_y2024m02 creada o ya existe
   ✓ Partición green_taxi_trip_y2024m03 creada o ya existe
   ✓ Partición green

##  Ingesta Yellow

In [ ]:
# INGESTA YELLOW TAXI A POSTGRESQL - Con particionamiento automático
import os
import time
from datetime import datetime, timezone
from pyspark.sql import functions as F
from pyspark.sql import types as T
import psycopg2

def normalize_yellow_columns(df):
    """Normaliza nombres de columnas inconsistentes en archivos yellow"""
    column_mapping = {
        'Airport_fee': 'airport_fee',
        'AIRPORT_FEE': 'airport_fee',
    }
    for old_name, new_name in column_mapping.items():
        if old_name in df.columns:
            df = df.withColumnRenamed(old_name, new_name)
    return df

# ============================================================================
# CONFIGURACIÓN POSTGRESQL DESDE VARIABLES DE ENTORNO
# ============================================================================
PG_HOST = os.getenv('PG_HOST', 'localhost')
PG_PORT = int(os.getenv('PG_PORT', 5432))
PG_DB = os.getenv('PG_DB', 'postgres')
PG_USER = os.getenv('PG_USER', 'postgres')
PG_PASSWORD = os.getenv('PG_PASSWORD', 'root')
PG_SCHEMA_RAW = os.getenv('PG_SCHEMA_RAW', 'raw')

print(f"📊 Conectando a PostgreSQL:")
print(f"   Host: {PG_HOST}:{PG_PORT}")
print(f"   Database: {PG_DB}")
print(f"   User: {PG_USER}")
print(f"   Schema: {PG_SCHEMA_RAW}")

# Configuración
service_types = ['yellow']
start_year = 2025
end_year = 2025
months = [9]
data_dir = './work/data'

run_id = f"raw_yellow_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}"
ingested_at_utc = datetime.now(timezone.utc).isoformat()

# Configuración PostgreSQL JDBC
pgOptions = {
    "url": f"jdbc:postgresql://{PG_HOST}:{PG_PORT}/{PG_DB}",
    "driver": "org.postgresql.Driver",
    "user": PG_USER,
    "password": PG_PASSWORD,
    "dbtable": f"{PG_SCHEMA_RAW}.yellow_taxi_trip"
}

# Función para crear partición si no existe
def create_partition_if_not_exists(year, month):
    """Crea la partición para año-mes si no existe"""
    partition_name = f"yellow_taxi_trip_y{year}m{month:02d}"
    
    # Calcular límites de la partición
    next_month = month + 1
    next_year = year
    if next_month > 12:
        next_month = 1
        next_year += 1
    
    create_partition_sql = f"""
    CREATE TABLE IF NOT EXISTS {PG_SCHEMA_RAW}.{partition_name} 
    PARTITION OF {PG_SCHEMA_RAW}.yellow_taxi_trip
    FOR VALUES FROM ({year}, {month}) TO ({next_year}, {next_month});
    """
    
    try:
        conn = psycopg2.connect(
            dbname=PG_DB,
            user=PG_USER,
            password=PG_PASSWORD,
            host=PG_HOST,
            port=PG_PORT
        )
        conn.autocommit = True
        cur = conn.cursor()
        cur.execute(create_partition_sql)
        cur.close()
        conn.close()
        print(f"   ✓ Partición {partition_name} creada o ya existe")
        return True
    except Exception as e:
        print(f"   ERROR creando partición {partition_name}: {e}")
        return False

# Esquema de la tabla principal PARTICIONADA
postgres_schema_yellow = f'''
CREATE SCHEMA IF NOT EXISTS {PG_SCHEMA_RAW};

CREATE TABLE IF NOT EXISTS {PG_SCHEMA_RAW}.yellow_taxi_trip (
    run_id VARCHAR,
    service_type VARCHAR,
    source_year INT NOT NULL,
    source_month INT NOT NULL,
    ingested_at_utc VARCHAR,
    source_path VARCHAR,
    VendorID INT,
    tpep_pickup_datetime TIMESTAMP,
    tpep_dropoff_datetime TIMESTAMP,
    passenger_count INT,
    trip_distance FLOAT,
    RatecodeID INT,
    store_and_fwd_flag VARCHAR,
    PULocationID INT,
    DOLocationID INT,
    payment_type INT,
    fare_amount FLOAT,
    extra FLOAT,
    mta_tax FLOAT,
    tip_amount FLOAT,
    tolls_amount FLOAT,
    improvement_surcharge FLOAT,
    total_amount FLOAT,
    congestion_surcharge FLOAT,
    airport_fee FLOAT,
    cbd_congestion_fee FLOAT
) PARTITION BY RANGE (source_year, source_month);

-- Crear índice en la tabla particionada (se propagará a las particiones)
CREATE INDEX IF NOT EXISTS idx_yellow_taxi_year_month 
ON {PG_SCHEMA_RAW}.yellow_taxi_trip (source_year, source_month);
'''

# Crear la tabla particionada
try:
    conn = psycopg2.connect(
        dbname=PG_DB,
        user=PG_USER,
        password=PG_PASSWORD,
        host=PG_HOST,
        port=PG_PORT
    )
    conn.autocommit = True
    cur = conn.cursor()
    cur.execute(postgres_schema_yellow)
    cur.close()
    conn.close()
    print(f"✓ Tabla particionada {PG_SCHEMA_RAW}.yellow_taxi_trip creada o ya existe")
except Exception as e:
    print(f"ERROR creando tabla: {e}")
    exit(1)

# Crear todas las particiones necesarias de antemano
print("\nCreando particiones necesarias...")
for year in range(start_year, end_year + 1):
    for month in months:
        create_partition_if_not_exists(year, month)

# Métricas globales
total_files_processed = 0
total_files_skipped = 0
total_records_read = 0
total_records_normalized = 0
total_records_transformed = 0
total_records_written = 0
total_records_deleted = 0
total_processing_time = 0

for service_type in service_types:
    print(f"\n{'='*80}")
    print(f"PROCESANDO {service_type.upper()} TAXI")
    print(f"{'='*80}")
    
    for year in range(start_year, end_year + 1):
        for month in months:
            fname = f'{service_type}_tripdata_{year}-{month:02d}.parquet'
            fpath = os.path.join(data_dir, fname)
            print(f"\nArchivo: {fname}")
            
            if not os.path.isfile(fpath):
                print(f"   Archivo no encontrado: {fname}")
                total_files_skipped += 1
                continue
            
            file_start_time = time.time()
            
            try:
                # PASO 1: Eliminar registros existentes del año/mes (idempotencia)
                print(f"   Verificando registros existentes para {year}-{month:02d}...")
                try:
                    conn = psycopg2.connect(
                        dbname=PG_DB,
                        user=PG_USER,
                        password=PG_PASSWORD,
                        host=PG_HOST,
                        port=PG_PORT
                    )
                    conn.autocommit = True
                    cur = conn.cursor()

                    # Contar registros existentes
                    cur.execute(
                        f"SELECT COUNT(*) FROM {PG_SCHEMA_RAW}.yellow_taxi_trip WHERE source_year = %s AND source_month = %s", 
                        (year, month)
                    )
                    existing_count = cur.fetchone()[0]
                    
                    if existing_count > 0:
                        print(f"   ⚠ Encontrados {existing_count:,} registros existentes - eliminando...")
                        cur.execute(
                            f"DELETE FROM {PG_SCHEMA_RAW}.yellow_taxi_trip WHERE source_year = %s AND source_month = %s", 
                            (year, month)
                        )
                        print(f"   ✓ {existing_count:,} registros eliminados")
                        total_records_deleted += existing_count
                    else:
                        print(f"   ✓ No hay registros previos para este periodo")
                    
                    cur.close()
                    conn.close()
                except Exception as e:
                    print(f"   ERROR verificando/eliminando registros: {e}")

                # PASO 2: Lectura del archivo
                df = spark.read.option("mergeSchema", "false").parquet(fpath)
                df = normalize_yellow_columns(df)
                
                records_read = df.count()
                print(f"   Registros leídos: {records_read:,}")
                
                if records_read == 0:
                    print("   Archivo vacío, saltando...")
                    total_files_skipped += 1
                    continue
                
                records_normalized = records_read
                
                # Verificar columnas opcionales
                has_cbd_fee = 'cbd_congestion_fee' in df.columns
                has_airport_fee = 'airport_fee' in df.columns
                print(f"   Columnas opcionales: airport_fee={has_airport_fee}, cbd_fee={has_cbd_fee}")
                
                # PASO 3: Transformaciones
                df_transformed = df.select(
                    F.lit(run_id).alias('run_id'),
                    F.lit(service_type).alias('service_type'),
                    F.lit(year).alias('source_year'),
                    F.lit(month).alias('source_month'),
                    F.lit(ingested_at_utc).alias('ingested_at_utc'),
                    F.lit(fpath).alias('source_path'),
                    F.col('VendorID').cast(T.IntegerType()).alias('VendorID'),
                    F.col('tpep_pickup_datetime').cast(T.TimestampType()).alias('tpep_pickup_datetime'),
                    F.col('tpep_dropoff_datetime').cast(T.TimestampType()).alias('tpep_dropoff_datetime'),
                    F.col('passenger_count').cast(T.IntegerType()).alias('passenger_count'),
                    F.col('trip_distance').cast(T.FloatType()).alias('trip_distance'),
                    F.col('RatecodeID').cast(T.IntegerType()).alias('RatecodeID'),
                    F.col('store_and_fwd_flag').cast(T.StringType()).alias('store_and_fwd_flag'),
                    F.col('PULocationID').cast(T.IntegerType()).alias('PULocationID'),
                    F.col('DOLocationID').cast(T.IntegerType()).alias('DOLocationID'),
                    F.col('payment_type').cast(T.IntegerType()).alias('payment_type'),
                    F.col('fare_amount').cast(T.FloatType()).alias('fare_amount'),
                    F.col('extra').cast(T.FloatType()).alias('extra'),
                    F.col('mta_tax').cast(T.FloatType()).alias('mta_tax'),
                    F.col('tip_amount').cast(T.FloatType()).alias('tip_amount'),
                    F.col('tolls_amount').cast(T.FloatType()).alias('tolls_amount'),
                    F.col('improvement_surcharge').cast(T.FloatType()).alias('improvement_surcharge'),
                    F.col('total_amount').cast(T.FloatType()).alias('total_amount'),
                    F.col('congestion_surcharge').cast(T.FloatType()).alias('congestion_surcharge'),
                    (F.col('airport_fee').cast(T.FloatType()) if has_airport_fee 
                     else F.lit(None).cast(T.FloatType())).alias('airport_fee'),
                    (F.col('cbd_congestion_fee').cast(T.FloatType()) if has_cbd_fee 
                     else F.lit(None).cast(T.FloatType())).alias('cbd_congestion_fee')
                )
                
                records_transformed = df_transformed.count()
                print(f"   Registros transformados: {records_transformed:,}")
                
                # PASO 4: Escritura JDBC a la partición correspondiente
                print(f"   Escribiendo a partición y{year}m{month:02d}...")
                (
                    df_transformed.write
                    .format("jdbc")
                    .options(**pgOptions)
                    .option("batchsize", 50000)
                    .option("numPartitions", 8)
                    .option("isolationLevel", "READ_UNCOMMITTED")
                    .mode("append")
                    .save()
                )
                
                df_transformed.unpersist()
                df.unpersist()
                
                records_written = records_transformed
                file_processing_time = time.time() - file_start_time
                
                # Actualizar métricas
                total_files_processed += 1
                total_records_read += records_read
                total_records_normalized += records_normalized
                total_records_transformed += records_transformed
                total_records_written += records_written
                total_processing_time += file_processing_time
                
                print(f"   ✓ Completado en {file_processing_time:.2f}s")
                print(f"   Velocidad: {records_written/file_processing_time:,.0f} reg/s")
                
                if records_read != records_written:
                    lost_records = records_read - records_written
                    loss_pct = (lost_records / records_read) * 100
                    print(f"   ⚠ Registros perdidos: {lost_records:,} ({loss_pct:.2f}%)")
                
                # Limpieza preventiva de caché
                if total_files_processed % 3 == 0:
                    spark.catalog.clearCache()
                    print("   Cache limpiado")
                    
            except Exception as e:
                print(f"   ERROR procesando {fname}: {e}")
                import traceback
                traceback.print_exc()
                total_files_skipped += 1
                continue

# RESUMEN FINAL
print(f"\n{'='*80}")
print(f"RESUMEN FINAL - YELLOW TAXI PARTICIONADO")
print(f"{'='*80}")
print(f"Run ID: {run_id}")
print(f"Timestamp UTC: {ingested_at_utc}")
print(f"\nCONFIGURACIÓN:")
print(f"   Método: PostgreSQL JDBC con particionamiento RANGE(year, month)")
print(f"   Particiones: {(end_year - start_year + 1) * len(months)} creadas")
print(f"   Paralelismo: 8 particiones Spark")
print(f"   Batch size: 50,000 registros")
print(f"\nARCHIVOS:")
print(f"   Procesados: {total_files_processed}")
print(f"   Omitidos: {total_files_skipped}")
print(f"   Total: {total_files_processed + total_files_skipped}")
print(f"\nREGISTROS:")
print(f"   Eliminados (previos): {total_records_deleted:,}")
print(f"   Leídos: {total_records_read:,}")
print(f"   Transformados: {total_records_transformed:,}")
print(f"   Escritos: {total_records_written:,}")
print(f"\nRENDIMIENTO:")
if total_processing_time > 0:
    print(f"   Tiempo total: {total_processing_time:.2f}s ({total_processing_time/60:.2f} min)")
    print(f"   Velocidad: {total_records_written/total_processing_time:,.0f} reg/s")
print(f"{'='*80}")


✓ Tabla particionada raw.yellow_taxi_trip creada o ya existe

Creando particiones necesarias...
   ✓ Partición yellow_taxi_trip_y2025m09 creada o ya existe

PROCESANDO YELLOW TAXI

Archivo: yellow_tripdata_2025-09.parquet
   Verificando registros existentes para 2025-09...
   ⚠ Encontrados 4,251,015 registros existentes - eliminando...
   ✓ 4,251,015 registros eliminados
   Registros leídos: 4,251,015
   Columnas opcionales: airport_fee=True, cbd_fee=True
   Registros transformados: 4,251,015
   Escribiendo a partición y2025m09...
   ✓ Completado en 475.11s
   Velocidad: 8,947 reg/s

RESUMEN FINAL - YELLOW TAXI PARTICIONADO
Run ID: raw_yellow_20251109_014814
Timestamp UTC: 2025-11-09T01:48:14.716710+00:00

CONFIGURACIÓN:
   Método: PostgreSQL JDBC con particionamiento RANGE(year, month)
   Particiones: 1 creadas
   Paralelismo: 8 particiones Spark
   Batch size: 50,000 registros

ARCHIVOS:
   Procesados: 1
   Omitidos: 0
   Total: 1

REGISTROS:
   Eliminados (previos): 4,251,015
   Leí